# WattWise AI - Step 1: Exploratory Data Analysis (London Smart Meters)
This notebook documents the ingestion, data quality verification, and consumption characteristics of the London Smart Meters dataset (UK Power Networks).

**Key Objectives:**
1. Missing-value summary and data-quality assessment
2. Average daily profile (diurnal shape, morning and evening peaks)
3. Weekday versus weekend behavioral comparisons
4. Electricity consumption versus ambient temperature (heating sensitivity)
5. Distribution of consumption across households

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load Processed Hourly and Daily Datasets

In [ ]:
df_hourly = pd.read_parquet('../data/processed/hourly.parquet')
df_daily = pd.read_parquet('../data/processed/daily.parquet')

print(f'Hourly records: {len(df_hourly):,}, Households: {df_hourly["household_id"].nunique()}')
print(f'Daily records:  {len(df_daily):,}, Households: {df_daily["household_id"].nunique()}')
df_hourly.head()

## 2. Missing-Value and Data-Quality Summary

In [ ]:
from src.preprocess import generate_data_quality_summary
summary = generate_data_quality_summary(df_hourly)
for k, v in summary.items():
    print(f'{k}: {v}')

## 3. Average Daily Profile (Diurnal Curve)

In [ ]:
df_hourly['hour'] = pd.to_datetime(df_hourly['timestamp']).dt.hour
hourly_profile = df_hourly.groupby('hour')['kwh'].agg(['mean', 'median', lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)])
hourly_profile.columns = ['mean', 'median', 'p25', 'p75']

plt.figure(figsize=(11, 4.5))
plt.plot(hourly_profile.index, hourly_profile['mean'], label='Mean Consumption', color='#1f77b4', lw=2.5)
plt.plot(hourly_profile.index, hourly_profile['median'], label='Median Consumption', color='#2ca02c', lw=2, linestyle='--')
plt.fill_between(hourly_profile.index, hourly_profile['p25'], hourly_profile['p75'], color='#1f77b4', alpha=0.15, label='IQR (25th - 75th percentile)')
plt.title('London Smart Meters: Average Diurnal Electricity Profile', fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=11)
plt.ylabel('Hourly Electricity (kWh)', fontsize=11)
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.show()

## 4. Weekday vs Weekend Comparison

In [ ]:
df_hourly['dayofweek'] = pd.to_datetime(df_hourly['timestamp']).dt.dayofweek
df_hourly['day_type'] = np.where(df_hourly['dayofweek'] >= 5, 'Weekend', 'Weekday')

day_profile = df_hourly.groupby(['day_type', 'hour'])['kwh'].mean().unstack(level=0)

plt.figure(figsize=(11, 4.5))
plt.plot(day_profile.index, day_profile['Weekday'], label='Weekday (Mon-Fri)', color='#1f77b4', lw=2.2)
plt.plot(day_profile.index, day_profile['Weekend'], label='Weekend (Sat-Sun)', color='#ff7f0e', lw=2.2, linestyle='-')
plt.title('Weekday vs Weekend Electricity Usage Comparison', fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=11)
plt.ylabel('Average Hourly Electricity (kWh)', fontsize=11)
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.show()

## 5. Consumption vs Outdoor Temperature

In [ ]:
plt.figure(figsize=(9, 5))
sns.regplot(data=df_daily.sample(min(len(df_daily), 5000), random_state=42), x='temperature', y='kwh', scatter_kws={'alpha': 0.25, 'color': '#3182bd'}, line_kws={'color': '#de2d26', 'lw': 2})
plt.title('Daily Electricity Consumption vs Ambient Temperature', fontsize=13, fontweight='bold')
plt.xlabel('Ambient Temperature (°C)', fontsize=11)
plt.ylabel('Daily Electricity Consumption (kWh)', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Distribution Across Households

In [ ]:
household_daily_mean = df_daily.groupby('household_id')['kwh'].mean()
plt.figure(figsize=(9, 4.5))
sns.histplot(household_daily_mean, bins=25, kde=True, color='#2b8cbe')
plt.title('Distribution of Average Daily Consumption Across Households', fontsize=13, fontweight='bold')
plt.xlabel('Mean Daily Electricity (kWh)', fontsize=11)
plt.ylabel('Household Count', fontsize=11)
plt.tight_layout()
plt.show()